In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from xgboost import XGBClassifier

DATASET = Path("Datasets/Processed/Dataset Dupilumab Baseline.xlsx - Anagrafica.csv")


def load_dataset():
    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / DATASET).is_file():
            return pd.read_csv(root / DATASET)
    raise FileNotFoundError(DATASET)

class MultiColumnBinarizer(BaseEstimator, TransformerMixin):
    # BaseEstimator: provides parameter handling for sklearn Pipelines and GridSearch
    # TransformerMixin: automatically implements the fit_transform() method

    def __init__(self, min_frequency):
        self.min_frequency = min_frequency
        self.binarizers_ = {}
        self.col_names_ = []
        self.frequent_classes_ = {}


    def _parse_series(self, series):
        series = series.fillna("").astype(str) # Missing values become 0
        parsed = []
        for row in series:
            if not row.strip():
                parsed.append([])
            else:
                items = [
                    item.strip().lower()
                    for item in row.split(",")
                    if item.strip()
                ]
                parsed.append(list(set(items)))
        return parsed

    def _map_rare_to_other(self, lists, freq_set):
        # Maps infrequent or unseen categories into 'other'
        return [list({item if item in freq_set else "other" for item in sublist}) for sublist in lists]

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.col_names_ = list(X_df.columns)
        self.binarizers_ = {}
        self.frequent_classes_ = {}

        for col in self.col_names_:
            lists = self._parse_series(X_df[col])
            
            # Count occurrences in the training set and keep classes >= min_frequency
            all_items = [item for sublist in lists for item in sublist]
            counts = pd.Series(all_items).value_counts()
            freq_set = set(counts[counts >= self.min_frequency].index)
            self.frequent_classes_[col] = freq_set

            # Map rare classes to 'other' before fitting MultiLabelBinarizer
            mapped_lists = self._map_rare_to_other(lists, freq_set)
            all_classes = sorted(list(freq_set | {"other"}))

            mlb = MultiLabelBinarizer(classes=all_classes, sparse_output=False)
            mlb.fit(mapped_lists)
            self.binarizers_[col] = mlb

        return self
   

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.col_names_)
        transformed_parts = []
        
        for col in self.col_names_:
            lists = self._parse_series(X_df[col])
            mapped_lists = self._map_rare_to_other(lists, self.frequent_classes_[col])
            transformed_parts.append(self.binarizers_[col].transform(mapped_lists))
            
        return np.hstack(transformed_parts)

    def get_feature_names_out(self, input_features=None):
        feature_names = []
        for col in self.col_names_:
            prefix = col.lower().replace(" ", "_")
            classes = self.binarizers_[col].classes_
            feature_names.extend([f"{prefix}_{c.replace(' ', '_')}" for c in classes])
        return np.array(feature_names)


In [ ]:
# XGBoost | set D
raw_xgb = load_dataset()

xgb_continuous_cols = [
    "Peso (kg)",
    "Altezza (cm)",
    "Neutrofili",
    "Linfociti",
    "Monociti",
    "Eosinofili",
    "Basofili",
    "IgE totali",
]

xgb_discrete_cols = [
    "Età",
    "Num Comorbidità",
    "Numero di pregressi interventi ESS",
    "N. allergeni positivi",
    "Giorni al mese di utilizzo spray",
    "ACT",
    "NPS",
    "Physical Symptoms",
    "Sleep Symptoms",
    "Emotional Symptoms",
    "Ostruzione nasale",
    "Rinorrea anteriore",
    "Rinorrea posteriore",
    "Starnuti",
    "QoL",
]

xgb_categorical_cols = [
    "Categoria Professione",
    "Sesso",
    "Biologico precedente (0 no, 1 sì)",
    "Familiarità Atopia",
    "OCS nell'ultimo anno",
    "Intolleranza ai FANS",
]

xgb_multi_categorical_cols = [
    "Comorbidità",
    "Quali allergeni positivi",
]

xgb_ordinal_cols = [
    "Fumo Baseline (0=no, 1=ex, 2=si)",
]

xgb_numeric = xgb_continuous_cols + xgb_discrete_cols + xgb_ordinal_cols
xgb_features = xgb_numeric + xgb_categorical_cols + xgb_multi_categorical_cols
X_xgb = raw_xgb.loc[:, xgb_features].copy()
y_xgb = raw_xgb["TARGET"].astype(int)

for col in xgb_numeric:
    values = X_xgb[col].astype("string").str.strip()
    european = values.str.contains(",", regex=False, na=False)
    values = values.mask(european, values.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    X_xgb[col] = pd.to_numeric(values, errors="coerce").astype(float)

xgb_pre = ColumnTransformer([
    ("num_cont", "passthrough", xgb_continuous_cols),
    ("num_discr", "passthrough", xgb_discrete_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(sparse_output=False, drop="if_binary", handle_unknown="ignore")),
    ]), xgb_categorical_cols),
    ("multi_cat", MultiColumnBinarizer(min_frequency=2), xgb_multi_categorical_cols),
    ("ord", "passthrough", xgb_ordinal_cols),
], verbose_feature_names_out=False)

# Compila i valori selezionati nella run del test.
xgb_model = XGBClassifier(
    n_estimators=None,
    max_depth=None,
    learning_rate=None,
    min_child_weight=None,
    random_state=None,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=1,
)
required = ("n_estimators", "max_depth", "learning_rate", "min_child_weight")
missing = [name for name in required if xgb_model.get_params()[name] is None]
if missing:
    raise ValueError("Compila i parametri XGBoost prima del fit: " + ", ".join(missing))

xgb_pipeline = Pipeline([("pre", xgb_pre), ("model", xgb_model)])
xgb_pipeline.fit(X_xgb, y_xgb)
